# German Corpus Analysis

This notebook is for analyzing the German sentences corpus.

**Prerequisite:** Before running this notebook, you must first run the data download script from your terminal:
```bash
python scripts/download_data.py
```
This will ensure the necessary `deu_mixed-typical_2011_1M-sentences.txt` file is present in the `data/deu_mixed-typical_2011_1M` directory.

## 1. Word Frequency Analysis

Now we will process the text file to count word occurrences.

We will use `spaCy` for tokenization and lemmatization, which allows us to group different forms of a word together (e.g., 'laufen', 'läufst', 'liefen' all become 'laufen'). We will also filter out common "stop words", punctuation, and numbers to get a cleaner frequency count.

In [ ]:
import os
import spacy
from collections import Counter
import pandas as pd
from tqdm import tqdm

# --- Configuration ---
DATA_DIR = "../data"
EXTRACTED_DIR = "deu_mixed-typical_2011_100k"
EXTRACTED_FILE_NAME = "deu_mixed-typical_2011_100k-sentences.txt"
FILE_PATH = os.path.join(DATA_DIR, EXTRACTED_DIR, EXTRACTED_FILE_NAME)

In [ ]:
# --- Load the spaCy model ---
# This might take a moment.
# If you get an error, run: python -m spacy download de_core_news_sm
try:
    nlp = spacy.load("de_core_news_sm")
except OSError:
    print("spaCy German model 'de_core_news_sm' not found.")
    print("Please run 'python -m spacy download de_core_news_sm' in your terminal.")
    raise

In [ ]:
# --- Process the file and count word lemmas ---
# We use two Counters to store the frequency of each lemma.
word_counts_with_stopwords = Counter()
word_counts_without_stopwords = Counter()

# We'll process the file line by line to handle large files efficiently.
print(f"Processing '{FILE_PATH}'...")
try:
    with open(FILE_PATH, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc="Counting words"):
            # The first part of the line is the index, followed by a tab.
            # We split the line at the first tab and take the second part.
            parts = line.split('\t', 1)
            if len(parts) == 2:
                sentence = parts[1]
                
                # Process the sentence with spaCy
                doc = nlp(sentence)
                
                # We iterate through each token in the processed sentence
                for token in doc:
                    # We only want to count actual words (not punctuation or numbers)
                    if token.is_alpha:
                        lemma = token.lemma_.lower()
                        # Add to the list that includes stop words
                        word_counts_with_stopwords[lemma] += 1
                        
                        # Add to the list that excludes stop words
                        if not token.is_stop:
                            word_counts_without_stopwords[lemma] += 1

except FileNotFoundError:
    print(f"Error: The file '{FILE_PATH}' was not found.")
    print("Please make sure you have run the 'scripts/download_data.py' script first.")

print("Processing complete.")

Processing '../data/deu_mixed-typical_2011_1M/deu_mixed-typical_2011_1M-sentences.txt'...


Counting words: 999926it [54:22, 306.46it/s]

Processing complete.


In [ ]:
# --- Display the most common words ---
print("Most common words (including stop words):")
for word, count in word_counts_with_stopwords.most_common(10):
    print(f"{word}: {count}")

print("\nMost common words (excluding stop words):")
for word, count in word_counts_without_stopwords.most_common(10):
    print(f"{word}: {count}")


Top 20 most common words:
prozent: 13941
stehen: 12758
liegen: 11579
euro: 11364
finden: 10071
bleiben: 8335
sehen: 6540
deutsch: 6404
geben: 5834
beginnen: 5639
leben: 5084
gelten: 4644
steigen: 4636
stellen: 4635
erfolgen: 4634
mensch: 4613
betragen: 4472
erhalten: 4456
führen: 4410
gehören: 4405
zeigen: 4403
zudem: 4356
wichtig: 4350
fallen: 4307
kind: 4228
mann: 4226
neu: 4175
unternehmen: 4159
hoch: 4155
kosten: 4135
seite: 4066
polizei: 4062
yahoo: 3889
halten: 3858
sprechen: 3809
punkt: 3787
bestehen: 3786
alt: 3760
einfach: 3748
spielen: 3724
million: 3719
bieten: 3709
bringen: 3619
nehmen: 3605
problem: 3584
frau: 3580
stadt: 3565
bild: 3553
setzen: 3508
derzeit: 3457
laufen: 3447
nächster: 3445
ergebnis: 3445
mal: 3431
erwarten: 3419
stark: 3379
frage: 3347
preis: 3334
brauchen: 3316
insgesamt: 3242
fall: 3221
entstehen: 3193
bitte: 3142
weg: 3132
ebenfalls: 3121
groß: 3118
land: 3107
legen: 3066
arbeit: 3046
kommen: 3044
letzter: 3030
spiel: 3025
haus: 3018
treffen: 2956
su

In [ ]:
# --- Convert to DataFrame and save to CSV ---
# We create a DataFrame from the word counts.
df_with_stopwords = pd.DataFrame(word_counts_with_stopwords.items(), columns=['word', 'count'])
df_without_stopwords = pd.DataFrame(word_counts_without_stopwords.items(), columns=['word', 'count'])

# Sort by count in descending order
df_with_stopwords = df_with_stopwords.sort_values(by='count', ascending=False)
df_without_stopwords = df_without_stopwords.sort_values(by='count', ascending=False)

# Define the output file paths
output_csv_with_stopwords = os.path.join(DATA_DIR, 'word_counts_with_stopwords.csv')
output_csv_without_stopwords = os.path.join(DATA_DIR, 'word_counts_without_stopwords.csv')

# Save to CSV
df_with_stopwords.to_csv(output_csv_with_stopwords, index=False)
df_without_stopwords.to_csv(output_csv_without_stopwords, index=False)

print(f"Word counts with stop words saved to '{output_csv_with_stopwords}'")
print(f"Word counts without stop words saved to '{output_csv_without_stopwords}'")


Word counts saved to '../data/word_counts.csv'
